# FastWoe Finetuning (WOE Recalibration)

Author: https://www.github.com/xRiskLab

WOE recalibration is a middle ground in credit risk between full redevelopment
(rebuild everything) and simple recalibration (adjust intercept/scaling).
The bins stay fixed, the likelihood ratios inside them shift.

The `finetune(X_new, y_new)` method updates WOE values using new data
while preserving the original bin structure. Pass a full DataFrame to
update all features, or a single-column DataFrame to update just one.

In [1]:
import numpy as np
import pandas as pd

from fastwoe import FastWoe

## 1. Fit on Original Data

Simulate a credit scoring dataset with two features:

- `grade` — categorical (loan grade)
- `income` — numerical (applicant income, will be binned automatically)


In [ ]:
np.random.seed(42)
n = 1000

grade = np.random.choice(["A", "B", "C", "D"], size=n, p=[0.4, 0.3, 0.2, 0.1])
income = np.random.lognormal(mean=10.5, sigma=0.5, size=n)

# Event rates vary by grade
default_prob = np.where(
    grade == "A", 0.05, np.where(grade == "B", 0.10, np.where(grade == "C", 0.20, 0.35))
)
y_train = np.random.binomial(1, default_prob)

X_train = pd.DataFrame({"grade": grade, "income": income})
y_train = pd.Series(y_train)

print(f"Training set: {len(X_train)} obs, event rate = {y_train.mean():.3f}")
X_train.head()

Training set: 1000 obs, event rate = 0.120


,grade,income
0,A,39689.839442
1,D,18626.262397
2,C,43918.851477
3,B,49280.949853
4,A,48045.090001


In [3]:
woe = FastWoe(
    binning_method="tree",
    tree_kwargs={"max_depth": 2},
    numerical_threshold=10,
    random_state=42,
)
woe.fit(X_train, y_train)

print("Original WOE mapping for 'grade':")
woe.get_mapping("grade")

Original WOE mapping for 'grade':


,category,count,count_pct,good_count,bad_count,event_rate,woe,woe_se,woe_ci_lower,woe_ci_upper
0,A,421,42.1,401,20,0.047506,-1.005799,0.229115,-1.454856,-0.556741
1,B,291,29.1,263,28,0.096220,-0.247519,0.198788,-0.637136,0.142097
2,C,188,18.8,146,42,0.223404,0.746493,0.175097,0.403310,1.089676
3,D,100,10.0,70,30,0.300000,1.145132,0.218218,0.717433,1.572831


In [4]:
print("Original WOE mapping for 'income':")
woe.get_mapping("income")

Original WOE mapping for 'income':


,category,count,count_pct,good_count,bad_count,event_rate,woe,woe_se,woe_ci_lower,woe_ci_upper
0,"(-∞, 19682.5]",89,8.9,82,7,0.078652,-0.468379,0.393767,-1.240148,0.303390
1,"(19682.5, 21198.9]",22,2.2,22,0,0.000000,-14.731802,0.621261,-15.949450,-13.514153
2,"(21198.9, 21280.1]",2,0.2,0,2,0.999996,14.326337,1.000000,12.366373,16.286301
3,"(21280.1, ∞)",887,88.7,776,111,0.125141,0.047808,0.101477,-0.151084,0.246700


In [5]:
print("Original feature stats:")
woe.get_feature_stats()

Original feature stats:


,feature,n_categories,total_observations,missing_count,missing_rate,gini,iv,iv_se,iv_ci_lower,iv_ci_upper,min_woe,max_woe
0,grade,4,1000,0,0.0,0.424697,0.639526,0.108817,0.426249,0.852803,-1.005799,1.145132
1,income,4,1000,0,0.0,0.076004,0.257158,0.169943,0.000000,0.590239,-14.731802,14.326337


## 2. Simulate Population Shift

A year later, the population has shifted: event rates for each grade have increased (economic downturn).

The bins (grades) are still the same, but the risk within each bin has changed.


In [ ]:
np.random.seed(123)
n_new = 800

grade_new = np.random.choice(["A", "B", "C", "D"], size=n_new, p=[0.35, 0.30, 0.25, 0.10])
income_new = np.random.lognormal(mean=10.4, sigma=0.5, size=n_new)

# Shifted event rates — higher defaults across the board
default_prob_new = np.where(
    grade_new == "A", 0.10, np.where(grade_new == "B", 0.18, np.where(grade_new == "C", 0.30, 0.45))
)
y_new = pd.Series(np.random.binomial(1, default_prob_new))

X_new = pd.DataFrame({"grade": grade_new, "income": income_new})

print(f"New data: {len(X_new)} obs, event rate = {y_new.mean():.3f} (was {y_train.mean():.3f})")

New data: 800 obs, event rate = 0.194 (was 0.120)


## 3. Finetune a Single Feature

Pass a single-column DataFrame to update just one feature.
The bin structure (A, B, C, D) stays the same — only the evidence shifts.

In [ ]:
# Save original values for comparison
original_grade_woe = woe.get_mapping("grade").set_index("category")["woe"].copy()

# Finetune just 'grade' — pass single-column DataFrame
woe.finetune(X_new[["grade"]], y_new)

print("Updated WOE mapping for 'grade':")
woe.get_mapping("grade")

In [ ]:
# Compare old vs new WOE
updated_grade_woe = woe.get_mapping("grade").set_index("category")["woe"]

comparison = pd.DataFrame(
    {
        "original_woe": original_grade_woe,
        "finetuned_woe": updated_grade_woe,
        "delta": updated_grade_woe - original_grade_woe,
    }
)
print("WOE shift after finetuning 'grade':")
comparison.round(4)

WOE shift after finetuning 'grade':


,original_woe,finetuned_woe,delta
category,,,
A,-1.0058,-0.7312,0.2746
B,-0.2475,0.4654,0.7129
C,0.7465,1.1815,0.4350
D,1.1451,1.5775,0.4324


## 4. Finetune All Features at Once

Pass the full DataFrame to update every feature in one call.
Bin edges are preserved — only the WOE within each bin is updated.

In [ ]:
original_income_woe = woe.get_mapping("income").set_index("category")["woe"].copy()
original_bin_edges = woe.binning_info_["income"]["bin_edges"].copy()

# Finetune all features — pass the full DataFrame
woe.finetune(X_new, y_new)

print("Updated WOE mapping for 'income':")
woe.get_mapping("income")

In [ ]:
# Verify bin edges are unchanged
assert woe.binning_info_["income"]["bin_edges"] == original_bin_edges
print("Bin edges unchanged:", original_bin_edges)

## 5. Transform Uses Updated WOE Values

After finetuning, `transform()` automatically uses the updated WOE mapping.


In [ ]:
X_woe = woe.transform(X_new)

print("Transformed output (first 5 rows):")
X_woe.head()

## 6. Updating the Global Prior

By default, `finetune()` keeps the original prior (`y_prior_` / `odds_prior_`)
so that other features remain consistent. Use `update_prior=True` when you
want to shift the baseline as well — but note that this makes all other
features' WOE values stale.


In [ ]:
import warnings

print(f"Prior before: {woe.y_prior_:.4f}")

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    woe.finetune(X_new[["grade"]], y_new, update_prior=True)
    for warning in w:
        print(f"Warning: {warning.message}")

print(f"Prior after:  {woe.y_prior_:.4f}")

## 7. Handling Missing Categories

If the new data doesn't contain all original bins, `finetune()` warns
and retains the original WOE for missing bins.


In [ ]:
# New data with only grades A and B (no C or D)
X_partial = pd.DataFrame({"grade": ["A"] * 100 + ["B"] * 100})
y_partial = pd.Series(np.random.binomial(1, 0.15, 200))

# Save current D WOE before finetune
d_woe_before = woe.mappings_["grade"].loc["D", "woe"]

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    woe.finetune(X_partial, y_partial)
    for warning in w:
        print(f"Warning: {warning.message}")

# D retains its previous WOE
d_woe_after = woe.mappings_["grade"].loc["D", "woe"]
print(f"\nGrade D WOE before: {d_woe_before:.4f}")
print(f"Grade D WOE after:  {d_woe_after:.4f} (unchanged)")